In [1]:
pip install kaggle

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score , roc_curve
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

In [3]:
# --- Upload & install Kaggle CLI, place creds, and test auth (all-in-one) ---
from google.colab import files
uploaded = files.upload()  # choose kaggle.json

import os, json, stat, subprocess, sys

# 1) Ensure the config dir exists
cfg_dir = "/root/.config/kaggle"
os.makedirs(cfg_dir, exist_ok=True)

# 2) Write the uploaded kaggle.json into the expected path
# (uploaded is a dict: {filename: bytes})
fname = next(iter(uploaded))  # should be 'kaggle.json'
kpath = os.path.join(cfg_dir, "kaggle.json")
with open(kpath, "wb") as f:
    f.write(uploaded[fname])

# 3) Set strict permissions (0600)
os.chmod(kpath, stat.S_IRUSR | stat.S_IWUSR)

# 4) Install kaggle CLI
!pip -q install kaggle

# 5) (Optional but robust) also export env vars from kaggle.json
with open(kpath) as f:
    creds = json.load(f)
os.environ["KAGGLE_USERNAME"] = creds.get("username", "")
os.environ["KAGGLE_KEY"] = creds.get("key", "")

# 6) Test: list competitions (auth must succeed)
print("Testing Kaggle CLI auth...")
!kaggle competitions list | head


Saving kaggle.json to kaggle.json
Testing Kaggle CLI auth...
ref                                                                              deadline             category                reward  teamCount  userHasEntered  
-------------------------------------------------------------------------------  -------------------  ---------------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/arc-prize-2025                               2025-11-03 23:59:00  Featured         1,000,000 Usd        985           False  
https://www.kaggle.com/competitions/jigsaw-agile-community-rules                 2025-10-24 06:59:00  Featured           100,000 Usd       1377           False  
https://www.kaggle.com/competitions/mitsui-commodity-prediction-challenge        2025-10-06 23:59:00  Featured           100,000 Usd       1052           False  
https://www.kaggle.com/competitions/google-code-golf-2025                        2025-10-30 23:59:00  Research           100,000 

In [4]:
# Download dataset zip
!kaggle competitions download -c jane-street-real-time-market-data-forecasting

# Unzip into a folder
!unzip -q -o jane-street-real-time-market-data-forecasting.zip -d jane_street_data

# Check files
!ls -lh jane_street_data


100% 11.4G/11.5G [00:22<00:00, 712MB/s]
100% 11.5G/11.5G [00:22<00:00, 556MB/s]
total 36K
-rw-r--r--  1 root root 8.7K Mar  5  2025 features.csv
drwxr-xr-x  3 root root 4.0K Sep 12 21:41 kaggle_evaluation
drwxr-xr-x  3 root root 4.0K Sep 12 21:41 lags.parquet
-rw-r--r--  1 root root  403 Mar  5  2025 responders.csv
-rw-r--r--  1 root root  282 Mar  5  2025 sample_submission.csv
drwxr-xr-x  3 root root 4.0K Sep 12 21:41 test.parquet
drwxr-xr-x 12 root root 4.0K Sep 12 21:42 train.parquet


In [5]:
!pip -q install lightgbm pyarrow pandas numpy scikit-learn joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 42.5 MB/s eta 0:00:00


In [7]:
!pip -q install fastparquet

import pandas as pd

DATA_DIR = "jane_street_data"

train = pd.read_parquet(f"{DATA_DIR}/train.parquet")  # pyarrow is fine here
lags  = pd.read_parquet(f"{DATA_DIR}/lags.parquet", engine="fastparquet")  # <-- key change

# normalize dtypes for the join
train["date_id"] = train["date_id"].astype("int32")
train["symbol_id"] = train["symbol_id"].astype("int32")
lags["date_id"] = lags["date_id"].astype("int32")
lags["symbol_id"] = lags["symbol_id"].astype("int32")

# merge lagged responders onto train
train = train.merge(
    lags.add_suffix("_lag"),
    left_on=["date_id", "symbol_id"],
    right_on=["date_id_lag", "symbol_id_lag"],
    how="left"
).drop(columns=["date_id_lag","symbol_id_lag"])

print("OK. train shape:", train.shape)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 86.9 MB/s eta 0:00:00
OK. train shape: (47127338, 103)


In [8]:
# ========= 2) Build features, holdout split by date, train LightGBM =========
import numpy as np, pandas as pd, gc
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import joblib

# --- columns ---
feature_cols = [f"feature_{i:02d}" for i in range(79)]
lag_cols     = [c for c in train.columns if c.startswith("responder_") and c.endswith("_lag")]
target_col   = "responder_6"

# Keep only what we need to cut memory
use_cols = feature_cols + lag_cols + [target_col, "date_id", "weight"]
df = train[use_cols].copy()
del train  # free RAM
gc.collect()

# Types + NaNs
for c in feature_cols + lag_cols:
    if c in df.columns:
        df[c] = df[c].astype("float32").fillna(0.0)
df[target_col] = df[target_col].astype("float32")
df["weight"]   = df["weight"].astype("float32")
df["date_id"]  = df["date_id"].astype("int32")

# Drop rows without target (should be none in train)
df = df.dropna(subset=[target_col]).reset_index(drop=True)

# ---- time-ordered split: last N dates as validation (adjust N if needed) ----
unique_dates = np.sort(df["date_id"].unique())
N_VAL_DATES = 20                      # increase for stricter validation; decrease if RAM is tight
val_dates = set(unique_dates[-N_VAL_DATES:])

train_df = df[~df["date_id"].isin(val_dates)]
valid_df = df[df["date_id"].isin(val_dates)]
print("Train rows:", len(train_df), " Valid rows:", len(valid_df))

# Optional: downsample training to fit memory / speed (tune as needed)
MAX_TRAIN_ROWS = 8_000_000            # raise if you have RAM, lower if OOM
if len(train_df) > MAX_TRAIN_ROWS:
    train_df = train_df.sample(n=MAX_TRAIN_ROWS, random_state=42)

X_tr = train_df[feature_cols + lag_cols]
y_tr = train_df[target_col]
w_tr = train_df["weight"]

X_va = valid_df[feature_cols + lag_cols]
y_va = valid_df[target_col]
w_va = valid_df["weight"]

# Free the big df frames
del df
gc.collect()

# ---- LightGBM params (CPU) ----
params = dict(
    objective="rmse",
    learning_rate=0.05,
    num_leaves=127,
    min_data_in_leaf=256,
    feature_fraction=0.85,
    bagging_fraction=0.85,
    bagging_freq=1,
    n_estimators=10_000,
    reg_alpha=0.1,
    reg_lambda=0.1,
    verbosity=-1,
    random_state=42,
)

model = lgb.LGBMRegressor(**params)
model.fit(
    X_tr, y_tr,
    sample_weight=w_tr,
    eval_set=[(X_va, y_va)],
    eval_sample_weight=[w_va],
    eval_metric="rmse",
    callbacks=[lgb.early_stopping(300), lgb.log_evaluation(200)]
)

# Weighted RMSE (to align with scoring using 'weight')
def weighted_rmse(y_true, y_pred, w):
    return np.sqrt(np.average((y_true - y_pred) ** 2, weights=w))

pred_va = model.predict(X_va, num_iteration=model.best_iteration_)
val_wrmse = weighted_rmse(y_va.values, pred_va, w_va.values)
print(f"Validation weighted RMSE: {val_wrmse:.6f}")

# Save artifacts
joblib.dump(model, "lgbm_forecast.pkl")
print("Saved model: lgbm_forecast.pkl")


Train rows: 46375202  Valid rows: 752136
Training until validation scores don't improve for 300 rounds
[200]	valid_0's rmse: 0.720741
[400]	valid_0's rmse: 0.720797
Early stopping, best iteration is:
[295]	valid_0's rmse: 0.720678
Validation weighted RMSE: 0.720678
Saved model: lgbm_forecast.pkl
